***

* [总目录](../0_Introduction/0_introduction.ipynb)
* [术语表](../0_Introduction/1_glossary.ipynb)
* [8. 校准](8_0_introduction.ipynb)
    * 上一节： [第 8 章：校准](8_0_introduction.ipynb)
    * 下一节： [8.2 第一代校准（1GC）](8_2_1gc.ipynb)

***


## 8.1 作为最小二乘问题的校准 <a id='cal:sec:cal_ls'></a>

校准之所以能够被数值求解，是因为观测数据和模型预测之间存在明确的测量方程。对单极化、方向无关的情形，可写成

$$
d_{pq}(t)=g_p(t)\,m_{pq}(t)\,g_q^*(t)+n_{pq}(t),
$$

其中 $d_{pq}$ 是观测可见度，$m_{pq}$ 是由天空模型预测的理想可见度，$g_p$ 是天线复增益，$n_{pq}$ 包含热噪声与未建模误差。给定一组观测和模型，校准的目标就是寻找增益参数，使加权残差尽可能小。

这个问题的难点并不在于每条基线只有一个公式，而在于所有基线共享同一组天线增益。$N$ 面天线有 $N(N-1)/2$ 条基线，因此数据中存在冗余约束；正是这些冗余使天线增益可以从基线可见度中反推出。与此同时，增益只以 $g_pg_q^*$ 的组合出现，所以存在全局相位等规范自由度，必须通过参考天线或其他约束固定。


### 8.1.1 从测量方程到目标函数

若把所有时间、频率和基线上的复残差写成

$$
r_{pq}(t)=d_{pq}(t)-g_p(t)m_{pq}(t)g_q^*(t),
$$

最常用的目标函数是加权平方和

$$
\chi^2(\boldsymbol{\theta})=
\sum_{t,p<q}w_{pq}(t)\left|r_{pq}(t;\boldsymbol{\theta})\right|^2,
$$

其中 $\boldsymbol{\theta}$ 表示待求的实参数，例如各天线的对数振幅和相位，$w_{pq}$ 表示数据权重。若噪声是独立复高斯噪声，并且权重等于方差倒数，那么最小化 $\chi^2$ 等价于最大似然估计。真实数据中的 RFI、模型误差和相关噪声会破坏这个理想条件，但最小二乘仍然提供了校准求解器的基本骨架。

为了让几何关系和参数退化更直观，下面采用一个五天线东西向阵列作为示例。天空模型取为一维方向余弦轴上的三个点源，模型可见度为

$$
m_{pq}(u)=\sum_s I_s\exp(-2\pi i u l_s).
$$

这种设置刻意简化了成像部分，使注意力集中在增益求解本身。


![用于最小二乘校准示例的阵列与 uv 轨迹](figures/calibration_ls_array_uv_tracks.png)

**图 8.1.1** 一个简化东西向阵列及其地球自转综合产生的 $uv$ 轨迹。多条基线同时约束同一组天线增益，这是校准可解性的基础。


### 8.1.2 参考天线与全局相位退化

校准方程中存在一个重要的不变性：若所有天线增益同时乘以同一个相位因子 $e^{i\phi_0}$，则

$$
(g_pe^{i\phi_0})m_{pq}(g_qe^{i\phi_0})^*=g_pm_{pq}g_q^*.
$$

因此，任意一条基线上的预测可见度都不变，目标函数也不变。数据本身无法决定这个全局相位零点，求解器若不加约束，就会在一个平坦方向上漂移。常见做法是选择一面参考天线，把它的相位固定为零；这样并不是声明参考天线“没有相位误差”，而只是给整组相位解选定一个坐标原点。


![全局相位对最小二乘目标函数不可见](figures/calibration_ls_global_phase_degeneracy.png)

**图 8.1.2** 把所有天线增益同时乘以同一个相位因子时，$\chi^2$ 几乎保持不变。参考天线固定的是规范自由度，而不是额外的物理信息。


### 8.1.3 受污染数据与初始残差

在示例数据中，理想模型可见度先被每台天线随时间变化的复增益污染，然后加入热噪声。对某条基线而言，观测振幅不再等于模型振幅，相位也出现随时间变化的偏移。未校准数据与模型之间的残差既包含天线增益造成的可校正结构，也包含噪声和模型无法解释的成分。

这一区分非常重要。求解器只能用所给的参数化模型解释残差；若残差中混入方向依赖误差、RFI 或天空模型缺失，方向无关增益仍会尽力吸收它们。最小二乘算法本身并不会判断残差来源是否物理合理。


![时间变化复增益造成的可见度污染](figures/calibration_ls_corrupted_data.png)

**图 8.1.3** 左侧显示一面天线的真实增益轨迹，右侧显示一条基线上的模型可见度与受污染数据。振幅和相位偏移都由天线增益组合 $g_pg_q^*$ 写入基线数据。


### 8.1.4 迭代求解：局部线性化与阻尼

由于残差中含有 $g_pg_q^*$，校准目标函数对增益参数是非线性的。常见求解器会在当前参数附近把残差作一阶展开：

$$
\boldsymbol{r}(\boldsymbol{\theta}+\Delta\boldsymbol{\theta})
\approx
\boldsymbol{r}(\boldsymbol{\theta})+
\mathbf{J}\Delta\boldsymbol{\theta},
$$

其中 $\mathbf{J}$ 是残差对参数的雅可比矩阵。高斯-牛顿方法求解

$$
(\mathbf{J}^T\mathbf{W}\mathbf{J})\Delta\boldsymbol{\theta}
=-\mathbf{J}^T\mathbf{W}\boldsymbol{r},
$$

而 Levenberg-Marquardt 方法会加入阻尼项，使更新在远离最优点时更保守，在接近最优点时更接近高斯-牛顿步。实际射电软件中的校准器会使用更高效的解析雅可比、分块结构和时间/频率解间隔；但核心思想仍是局部线性化、解正规方程、更新参数、重新计算残差。


![LM 风格求解器恢复方向无关复增益](figures/calibration_ls_lm_gain_recovery.png)

**图 8.1.4** 简化 LM 求解器对每个时间片求解天线增益。平均残差快速下降，求得的天线振幅和相位轨迹接近真实相对增益，改正后的基线可见度回到模型附近。


### 8.1.5 更换参考天线后的不变量

参考天线改变后，单台天线相位解的数值会整体重排，但改正后的可见度应该保持不变。若参考从天线 0 改为天线 2，天线 3 的相位曲线会变成相对于新参考的相位；然而每条基线上的校正因子仍应抵消观测中的 $g_pg_q^*$。因此，增益表的相位本身不能脱离参考约定解释，真正具有物理意义的是改正后的数据和与模型一致的闭合关系。


![参考天线改变解的表示而不改变校正结果](figures/calibration_ls_reference_antenna.png)

**图 8.1.5** 更换参考天线会改变天线相位解的表示方式，但两组解改正后的可见度几乎一致。参考天线选择固定的是规范，而不是独立观测量。


### 8.1.6 模型不完整时的误差吸收

最小二乘校准最容易被误用的地方，是把“目标函数下降”误认为“物理模型正确”。若天空模型漏掉一个弱源，求解器面对的是一个先天错误的问题。它仍会调整增益来减小残差，但这些增益不再只代表仪器响应，而会夹带未建模天空结构的信息。

这种误差吸收会带来两类后果。一方面，校准后的残差无法降到噪声水平附近，且常表现为与源位置或 $uv$ 覆盖相关的结构；另一方面，增益解可能看起来平滑而可信，却已经把真实天体结构的一部分压进了仪器项。高动态范围成像中，模型不完整、自校准过度和真实源结构被削弱，常常就是这样联系在一起的。


![天空模型不完整时求解器会吸收结构误差](figures/calibration_ls_incomplete_model.png)

**图 8.1.6** 完整模型和不完整模型下的求解行为。错误模型仍能让残差下降，但相对于真实天空的残差保持较高，说明求解器无法替代天空模型的物理正确性。


### 8.1.7 本节小结

方向无关校准可以被看作一个带规范约束的非线性最小二乘问题。数据冗余使天线增益可由基线可见度反推；参考天线固定全局相位自由度；迭代求解器通过局部线性化逐步降低残差；模型不完整则会让求解器把天空误差吸收到增益里。

这些结论直接通向下一节的 1GC。外场校准源之所以重要，是因为它在较简单、较可信的天空模型下提供通量、相位、延迟和带通标尺；而 1GC 的局限也由这里的数学结构决定：一旦真实误差不再能由方向无关 $g_p$ 描述，残差下降就不再等价于完整校准。
